In [ ]:
import pandas as pd
import numpy as np

import torch
import torch.nn as nn
from catboost import CatBoostRegressor
from sklearn.model_selection import KFold
from sklearn.metrics import mean_absolute_error

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
cd drive/MyDrive/

/content/drive/MyDrive


In [ ]:
train = pd.read_csv('traffic_V5.csv')
test = pd.read_csv('test_traffic_V5.csv')

print(f"학습 데이터 크기: {train.shape}")
print(f"테스트 데이터 크기: {test.shape}")

학습 데이터 크기: (249944, 94)
테스트 데이터 크기: (50000, 93)


In [ ]:
TARGET = 'avg_delay_minutes_next_30m'
ID_COLS = ['ID', 'layout_id', 'scenario_id']

feature_cols = [c for c in train.columns if c not in ID_COLS + [TARGET]]
print(f"피처 수: {len(feature_cols)}")

피처 수: 90


In [ ]:
kf = KFold(n_splits=5, shuffle=True, random_state=42)
oof_preds = np.zeros(len(train))
test_preds = np.zeros(len(test))

for fold, (tr_idx, val_idx) in enumerate(kf.split(train)):
    print(f"── Fold {fold + 1} ──")
    X_tr = train.loc[tr_idx, feature_cols]
    y_tr = train.loc[tr_idx, TARGET]
    X_val = train.loc[val_idx, feature_cols]
    y_val = train.loc[val_idx, TARGET]

    model = CatBoostRegressor(
        iterations=1000,
        learning_rate=0.05,
        depth=7,
        l2_leaf_reg=3,
        loss_function='MAE',
        eval_metric='MAE',
        random_seed=42,
        task_type='CPU',
        verbose=100
    )

    model.fit(
        X_tr, y_tr,
        eval_set=(X_val, y_val),
        early_stopping_rounds=50,
        use_best_model=True
    )

    oof_preds[val_idx] = model.predict(X_val)
    test_preds += model.predict(test[feature_cols]) / 5

── Fold 1 ──
0:	learn: 14.1947176	test: 14.2523039	best: 14.2523039 (0)	total: 196ms	remaining: 3m 15s
100:	learn: 9.3521522	test: 9.4421508	best: 9.4421508 (100)	total: 12.7s	remaining: 1m 53s
200:	learn: 9.2415015	test: 9.3690590	best: 9.3690590 (200)	total: 24.7s	remaining: 1m 38s
300:	learn: 9.1284611	test: 9.2974778	best: 9.2974778 (300)	total: 38s	remaining: 1m 28s
400:	learn: 9.0258721	test: 9.2366151	best: 9.2366151 (400)	total: 50.6s	remaining: 1m 15s
500:	learn: 8.9360111	test: 9.1847951	best: 9.1847951 (500)	total: 1m 3s	remaining: 1m 3s
600:	learn: 8.8580880	test: 9.1393896	best: 9.1393896 (600)	total: 1m 16s	remaining: 50.6s
700:	learn: 8.7869187	test: 9.0998600	best: 9.0998600 (700)	total: 1m 29s	remaining: 38.1s
800:	learn: 8.7249346	test: 9.0659783	best: 9.0659783 (800)	total: 1m 42s	remaining: 25.5s
900:	learn: 8.6656597	test: 9.0356122	best: 9.0356122 (900)	total: 1m 55s	remaining: 12.7s
999:	learn: 8.6094778	test: 9.0086827	best: 9.0086827 (999)	total: 2m 8s	remainin

In [ ]:
oof_mae = mean_absolute_error(train[TARGET], oof_preds)
print(f"OOF MAE: {oof_mae:.4f}")

OOF MAE: 8.9627


In [ ]:
submission = pd.DataFrame({'ID': test['ID'], TARGET: test_preds})
submission.to_csv('./submission_V19.csv', index=False)
print("submission.csv 저장 완료.")

submission.csv 저장 완료.
